## Camada Gold
<p>Na camada gold deste projeto, consumimos os dados gerados na camada Silver, gerando tabelas analíticas voltadas ao negócio</p>
Neste notebook, observamos
<ul>
<li>Agregações</li>
<li>Joins</li>
<li>Window functions</li>
<li>Consulta SQL</li>
<li>Pivotação</li>
</ul>

In [0]:
# Imports
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##### Definição de constantes

In [0]:
CATALOGO      = "ANP_Combustiveis"

silverSchema = "02_silver"
goldSchema   = "03_gold"

tablesSilver = {
    "PRECOS_REVENDA":   f"`{CATALOGO}`.`{silverSchema}`.`precos_revenda`",
    "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{silverSchema}`.`vendas_municipio`",
    "ESTADOS":          f"`{CATALOGO}`.`{silverSchema}`.`estados`"
}

tablesGold = {
    "INDICADORES_ESTADUAIS": f"`{CATALOGO}`.`{goldSchema}`.`indicadores_estaduais`"
}

fuelCategories = [
    "GASOLINA",
    "ETANOL",
    "DIESEL",
    "DIESEL S10",
    "GNV"
]

##### Definição da função auxiliar para atualizar tabela Dellta

In [0]:
# Função auxiliar para carregar dados na tabela Delta
def mergeToDelta(df: DataFrame, tableName: str, condition: str):
    deltaTable = DeltaTable.forName(spark, tableName)

    (
        deltaTable.alias("target")
        .merge(
            df.alias("source"),
            condition,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"[Merge-DeltaTable] Carga concluída em: {tableName}")

##### Leitura dos dados na camada Silver

In [0]:
# Leitura dos dados de preços da camada Silver
silverPricesDf = (
    spark.table(tablesSilver["PRECOS_REVENDA"])
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "id_revenda",
        "produto",
        "familia_combustivel",
        "data_coleta",
        "valor_venda",
    )
)

# Leitura dos dados de vendas da camada Silver
silverSalesDf = (
    spark.table(tablesSilver["VENDAS_MUNICIPIO"])
    .select(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
        "volume_vendido",
    )
)


### Análise das métricas de negócio

In [0]:
# Definição dos dados de análise
year = 2024                    # Ano que se deseja analisar
fuel = ["GASOLINA", "ETANOL"]  # Combustíveis que se deseja analisar

In [0]:
## Preparação dos dados

# Seleciona dados da tabela silver para o ano que se deseja realizar a análise
stateSilverDf = (
    spark.table(tablesSilver["ESTADOS"])
    .filter((F.col("ano_populacao") == year))
    .select(
        "codigo_uf",
        "uf",
        "nome_uf",
        "ano_populacao",
        "populacao",
        "ano_area",
        "area_km2"
    )
)

# Cálculo do preço médio estadual
statesAvgPriceDf = (
    silverPricesDf
    .filter(
        (F.year("data_coleta") == year)
        & F.col("familia_combustivel").isin(fuel)
    )
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia", "uf", "familia_combustivel")
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio")
    )
)

# Volume total vendido por estado
stateSalesDf = (
    silverSalesDf
    .filter(
        (F.col("ano_referencia") == year) &
        F.col("familia_combustivel").isin(fuel)
    )
    .groupBy("ano_referencia", "uf", "familia_combustivel",)
    .agg(
        F.sum("volume_vendido")
        .cast("double")
        .alias("volume_vendido")
    )
)

# Tabela unificada para [preço médio, volume vendido, população, area]
analysisBaseCalcDf = (
    stateSalesDf
    .join(statesAvgPriceDf, on=["ano_referencia", "uf", "familia_combustivel"], how="left")
    .join(stateSilverDf, on="uf", how="inner")
    # Cálculo do volume médio vendido por habitante
    .withColumn(
        "litros_por_habitante",
        F.round(F.col("volume_vendido") / F.col("populacao"), 3)
    )
    # Cálculo do volume médio vendido por km²
    .withColumn(
        "litros_por_km2",
        F.round(F.col("volume_vendido") / F.col("area_km2"), 3)
    )
)

# Janelas dos dados de para métrica nas tabelas
priceWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("preco_medio"))
)

volumeWindow  = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("volume_vendido"))
)

volumeByHabWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("litros_por_habitante"))
)

areaWindow = (
    Window
    .partitionBy("ano_referencia", "familia_combustivel")
    .orderBy(F.desc("litros_por_km2"))
)


# Ranking das métricas por Estado
stateMetricsDf = (
    analysisBaseCalcDf
    # Ranking por preço
    .withColumn("ranking_preco", F.dense_rank().over(priceWindow).cast("int"))
    # Ranking volume consumido
    .withColumn("ranking_consumo", F.dense_rank().over(volumeWindow).cast("int"))
    # Ranking volume consumido / hab
    .withColumn("ranking_consumo_habitante",F.dense_rank().over(volumeByHabWindow).cast("int"))
    # Ranking volume consumido / area
    .withColumn("ranking_consumo_area", F.dense_rank().over(areaWindow).cast("int"))
    .select(
        "ano_referencia",
        "codigo_uf",
        "uf",
        "nome_uf",
        "familia_combustivel",
        "preco_medio",
        "volume_vendido",
        "ano_populacao",
        "populacao",
        "ano_area",
        "area_km2",
        "litros_por_habitante",
        "litros_por_km2",
        "ranking_preco",
        "ranking_consumo",
        "ranking_consumo_habitante",
        "ranking_consumo_area",

        F.current_timestamp().alias("processado_gold_em")
    )
)


# Atualizando dados na tabela Delta
mergeToDelta(stateMetricsDf, tablesGold["INDICADORES_ESTADUAIS"],
    """
        target.ano_referencia
            = source.ano_referencia
        AND target.codigo_uf
            = source.codigo_uf
        AND target.familia_combustivel
            = source.familia_combustivel
    """
)

<h6>Métrica 1 - Avaliação da variação anual dos preços de combustível</h6>
<ul>
<li>Pivotação do tipo de combustivel, agrupado pela média anual do valor de venda.</li>
<li>Consulta dos valores através de spark SQL</li>
</ul>

In [0]:
pricesPivotDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia")
    .pivot("familia_combustivel", fuelCategories)
    .agg(F.round(F.avg("valor_venda"), 3))
    .orderBy("ano_referencia")
)

pricesPivotDf.createOrReplaceTempView(
    "vw_precos_medios_anuais"
)

pricesPivotSqlDf = spark.sql(
    """
    SELECT *
    FROM vw_precos_medios_anuais
    ORDER BY ano_referencia
    """
)

display(pricesPivotSqlDf)

<h6>Métrica 2.1 - Ranking de preço médio por estado</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "uf",
        "nome_uf",
        "preco_medio"
    )
    .orderBy("familia_combustivel", "ranking_preco")
)

<h6>Métrica 2.2 - Ranking estadual de volume vendido</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo",
        "uf",
        "nome_uf",
        "volume_vendido"
    )
    .orderBy("familia_combustivel", "ranking_consumo")
)

##### Métrica 2.3 — Ranking estadual de volume vendido por habitante

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo_habitante",
        "uf",
        "nome_uf",
        "volume_vendido",
        "populacao",
        "litros_por_habitante",
    )
    .orderBy(
        "familia_combustivel",
        "ranking_consumo_habitante",
    )
)

<h6>Métrica 2.3 - Ranking estadual de [volume vendido / km²]</h6>

In [0]:
display(
    stateMetricsDf
    .select(
        "ano_referencia",
        "familia_combustivel",
        "ranking_consumo_area",
        "uf",
        "nome_uf",
        "volume_vendido",
        "area_km2",
        "litros_por_km2"
    )
    .orderBy("familia_combustivel", "ranking_consumo_area")
)